# КИМ 5.1. Рекуррентные сети и 1D-CNN — эталонное решение

> **Это эталонное решение** для преподавателя. Студентам выдаётся ноутбук-задание
> [`kim-05-sequences.ipynb`](./kim-05-sequences.ipynb) без заполненных ячеек.
>
> Код ниже — один из возможных вариантов решения; не единственный и не обязательно
> оптимальный. Приводится для сверки и подготовки к защите.

---
## Часть А. Предобработка текстов

### 0. Импорт

In [ ]:
import numpy as np
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import pymorphy3

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

morph = pymorphy3.MorphAnalyzer()
ru_stop = set(stopwords.words('russian'))

### 1. Функция preprocess

In [ ]:
def preprocess(text):
    text = text.lower()
    tokens = word_tokenize(text, language='russian')
    tokens = [t for t in tokens if t.isalpha()]                      # без пунктуации/цифр
    tokens = [t for t in tokens if t not in ru_stop]                 # без стоп-слов
    tokens = [morph.parse(t)[0].normal_form for t in tokens]         # лемматизация
    return tokens

print(preprocess('Мама мыла раму, а кошка быстро бегала по двору!'))

### 2. Построение словаря по частоте

In [ ]:
from collections import Counter

def build_vocab(all_tokens, max_words=10000):
    freq = Counter()
    for toks in all_tokens:
        freq.update(toks)
    # 0 = pad, 1 = OOV, слова нумеруются с 2
    word_to_index = {'<PAD>': 0, '<OOV>': 1}
    for word, _ in freq.most_common(max_words - 2):
        word_to_index[word] = len(word_to_index)
    return word_to_index

# Загрузим отзывы (здесь — заглушка; на практике загрузить banks.csv)
# import pandas as pd
# df = pd.read_csv('banks.csv')
# texts = df['text'].tolist(); labels = df['label'].tolist()
# tokens_list = [preprocess(t) for t in texts]
# word_to_index = build_vocab(tokens_list, max_words=10000)
# print('Размер словаря:', len(word_to_index))
print('(заглушка — нужен banks.csv)')

### 3. text_to_sequence + pad_sequences

In [ ]:
def text_to_sequence(tokens, word_to_index, maxlen=200):
    seq = [word_to_index.get(t, 1) for t in tokens]   # 1 = OOV
    if len(seq) < maxlen:
        seq = [0] * (maxlen - len(seq)) + seq          # предпаддинг
    return seq[-maxlen:]

# Применить ко всем текстам:
# X = np.array([text_to_sequence(t, word_to_index, 200) for t in tokens_list])
# y = np.array(labels)
# from sklearn.model_selection import train_test_split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('(заглушка — нужен banks.csv)')

---
## Часть Б. Классификация текстов: LSTM vs 1D-CNN

### 4. LSTM-модель

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

max_words = 10000
model_lstm = keras.Sequential([
    layers.Embedding(max_words, 64, input_length=200),
    layers.LSTM(64),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])
model_lstm.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model_lstm.summary()
# hist_lstm = model_lstm.fit(X_train, y_train, batch_size=128, epochs=10,
#                            validation_split=0.1, verbose=2)

### 5. 1D-CNN-модель

In [ ]:
model_cnn = keras.Sequential([
    layers.Embedding(max_words, 64, input_length=200),
    layers.Conv1D(250, 5, padding='valid', activation='relu'),
    layers.GlobalMaxPooling1D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid'),
])
model_cnn.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model_cnn.summary()
# hist_cnn = model_cnn.fit(X_train, y_train, batch_size=128, epochs=10,
#                          validation_split=0.1, verbose=2)

### 6. Сравнение (после обучения)

| Модель | Val accuracy | Время/эпоха |
|---|---|---|
| LSTM | ~0.85–0.87 | ~30 с |
| 1D-CNN | ~0.83–0.85 | ~5 с |

**Вывод:** 1D-CNN обучается в ~6 раз быстрее LSTM и почти не уступает по точности
на коротких текстах. LSTM выигрывает на длинных зависимостях (где важен порядок
и контекст далеко от текущего слова). Для бинарной классификации отзывов
1D-CNN — отличный бейзлайн; LSTM — если нужна максимальная точность.

**Ответ (свёртка над эмбеддингами):** `Conv1D(k, ...)` извлекает
локальные n-граммы длины `k` (в коде — 5-граммы) из последовательности
эмбеддингов. `GlobalMaxPooling1D` выбирает самый «сильный» признак по всей
последовательности — то слово/n-грамму, которая сильнее всего активирует фильтр.
В отличие от LSTM, CNN не моделирует длинные зависимости и порядок дальше ядра,
но параллелится и быстра.

---
## Часть В. Временной ряд (Jena Climate)

### 7. Загрузка и подготовка

In [ ]:
import os
# Jena Climate загружается через keras.utils
zip_path = keras.utils.get_file(
    origin='https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip',
    fname='jena_climate_2009_2016.csv.zip',
    extract=True)
csv_path, _ = os.path.splitext(zip_path)

import pandas as pd
df = pd.read_csv(csv_path)
print(df.shape)
raw_data = df.iloc[:, 1:].values.astype('float32')   # без Date Time
temperature = raw_data[:, 1]                          # T (degC) — целевая

# Нормализация по train-части (первые 50%)
num_train = int(0.5 * len(raw_data))
num_val   = int(0.25 * len(raw_data))
mean = raw_data[:num_train].mean(axis=0)
std  = raw_data[:num_train].std(axis=0)
raw_data = (raw_data - mean) / std

### 8. Наивный бейзлайн

In [ ]:
sampling_rate = 6       # одно измерение в час (данные каждые 10 мин)
sequence_length = 120   # 5 дней истории

# Формируем датасеты
from tensorflow.keras.utils import timeseries_dataset_from_array
batch_size = 256

def make_ds(start, end):
    return timeseries_dataset_from_array(
        raw_data[:-sequence_length][start:end],
        targets=temperature[sequence_length:][start:end],
        sampling_rate=sampling_rate, sequence_length=sequence_length,
        batch_size=batch_size, shuffle=(start == 0))

train_ds = make_ds(0, num_train)
val_ds   = make_ds(num_train, num_train + num_val)
test_ds  = make_ds(num_train + num_val, None)

def evaluate_naive(dataset):
    abs_errs = []
    for samples, targets in dataset:
        preds = samples[:, -1, 1] * std[1] + mean[1]  # последнее значение T
        abs_errs.append(np.abs(preds - targets))
    return np.mean(np.concatenate(abs_errs))

print(f'Naive baseline MAE: {evaluate_naive(val_ds):.4f}')

### 9. LSTM-модель

In [ ]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.LSTM(16)(inputs)
outputs = layers.Dense(1)(inputs[:, -1, 1] if False else x)  # просто x
model = keras.Model(inputs, outputs)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

callbacks = [keras.callbacks.ModelCheckpoint('jena_lstm.keras', save_best_only=True)]
history = model.fit(train_ds, epochs=10, validation_data=val_ds, callbacks=callbacks)
model = keras.models.load_model('jena_lstm.keras')
print(f'LSTM val MAE: {model.evaluate(val_ds, verbose=0)[1]:.4f}')
print(f'LSTM test MAE: {model.evaluate(test_ds, verbose=0)[1]:.4f}')

### 10. Stacked GRU с recurrent_dropout

In [ ]:
inputs = keras.Input(shape=(sequence_length, raw_data.shape[-1]))
x = layers.GRU(32, recurrent_dropout=0.5, return_sequences=True)(inputs)
x = layers.GRU(32, recurrent_dropout=0.5)(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1)(x)
model_gru = keras.Model(inputs, outputs)
model_gru.compile(optimizer='adam', loss='mse', metrics=['mae'])
# model_gru.fit(train_ds, epochs=20, validation_data=val_ds)
print('(шаблон — запустить обучение при наличии ресурсов)')
# Ожидаемо: MAE LSTM ~0.25-0.28, Stacked GRU ~0.26-0.29, оба лучше бейзлайна (~0.29)

**Ответ (регрессия vs классификация временного ряда):** регрессия
прогнозирует непрерывное значение (температура), поэтому выходной слой — один
нейрон без активации (линейный), а loss — MSE/MAE. В классификации на выходе
`softmax` для вероятностей классов и loss — кросс-энтропия.

**Зачем наивный бейзлайн:** он задаёт нижнюю границу качества. Если
нейросеть не превосходит «прогноз последним значением», значит, она не
успела извлечь закономерность из истории — модель нужно усложнять или
менять подход.